In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jtrotman/formula-1-race-data")

print("Path to dataset files:", path)

100%|██████████| 6.64M/6.64M [00:00<00:00, 142MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/jtrotman/formula-1-race-data/versions/116


In [7]:
import pandas as pd
import os

from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

# ---------------------------------------
# 1. FIXED PATH DETECTION
# ---------------------------------------

BASE_PATH = "/root/.cache/kagglehub/datasets/jtrotman/formula-1-race-data/versions"

# go inside version folder (like 116)
version_folders = os.listdir(BASE_PATH)
DATA_PATH = os.path.join(BASE_PATH, version_folders[0])

print("Using dataset folder:", DATA_PATH)
print("Files:", os.listdir(DATA_PATH))

# ---------------------------------------
# 2. LOAD DATA
# ---------------------------------------

results = pd.read_csv(os.path.join(DATA_PATH, "results.csv"))
drivers = pd.read_csv(os.path.join(DATA_PATH, "drivers.csv"))
constructors = pd.read_csv(os.path.join(DATA_PATH, "constructors.csv"))
races = pd.read_csv(os.path.join(DATA_PATH, "races.csv"))

# ---------------------------------------
# 3. MERGE
# ---------------------------------------

df = results.merge(races, on="raceId")
df = df.merge(drivers, on="driverId")
df = df.merge(constructors, on="constructorId")

# ---------------------------------------
# 4. CLEAN
# ---------------------------------------

df = df[['year', 'round', 'driverId', 'constructorId',
         'grid', 'positionOrder', 'points']]

df['grid'] = pd.to_numeric(df['grid'], errors='coerce')
df['positionOrder'] = pd.to_numeric(df['positionOrder'], errors='coerce')
df['points'] = pd.to_numeric(df['points'], errors='coerce')

df['grid'] = df['grid'].fillna(20)
df = df.dropna()

# ---------------------------------------
# 5. TARGET
# ---------------------------------------

df['winner'] = (df['positionOrder'] == 1).astype(int)

# ---------------------------------------
# 6. SORT
# ---------------------------------------

df = df.sort_values(by=['year', 'round'])

# ---------------------------------------
# 7. FEATURES (NO LEAKAGE)
# ---------------------------------------

df['driver_avg_pos'] = df.groupby('driverId')['positionOrder']\
    .transform(lambda x: x.shift().rolling(5, min_periods=1).mean())

df['constructor_avg_pos'] = df.groupby('constructorId')['positionOrder']\
    .transform(lambda x: x.shift().rolling(5, min_periods=1).mean())

df = df.dropna()

features = ['grid', 'driver_avg_pos', 'constructor_avg_pos']

X = df[features].astype(float)
y = df['winner']

# ---------------------------------------
# 8. SPLIT
# ---------------------------------------

train = df[df['year'] < 2020]
test = df[df['year'] >= 2020]

X_train = train[features]
y_train = train['winner']

X_test = test[features]
y_test = test['winner']

# ---------------------------------------
# 9. IMBALANCE
# ---------------------------------------

scale = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

# ---------------------------------------
# 10. MODEL
# ---------------------------------------

model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale,
    eval_metric='logloss'
)

model.fit(X_train, y_train)

# ---------------------------------------
# 11. RESULTS
# ---------------------------------------

y_pred = model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Using dataset folder: /root/.cache/kagglehub/datasets/jtrotman/formula-1-race-data/versions/116
Files: ['constructor_results.csv', 'results.csv', 'pit_stops.csv', 'constructors.csv', 'seasons.csv', 'circuits.csv', 'drivers.csv', 'races.csv', 'constructor_standings.csv', 'driver_standings.csv', 'status.csv', 'sprint_results.csv', 'qualifying.csv', 'lap_times.csv']

Accuracy: 0.8323938369034197

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.83      0.90      2527
           1       0.22      0.90      0.35       134

    accuracy                           0.83      2661
   macro avg       0.61      0.86      0.63      2661
weighted avg       0.95      0.83      0.88      2661



In [8]:
# ---------------------------------------
# NEW TARGET (TOP 3)
# ---------------------------------------

df['top3'] = df['positionOrder'].apply(lambda x: 1 if x <= 3 else 0)

y = df['top3']

# ---------------------------------------
# SPLIT
# ---------------------------------------

train = df[df['year'] < 2020]
test = df[df['year'] >= 2020]

X_train = train[features]
y_train = train['top3']

X_test = test[features]
y_test = test['top3']

# ---------------------------------------
# IMBALANCE FIX
# ---------------------------------------

scale = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

# ---------------------------------------
# MODEL
# ---------------------------------------

model = XGBClassifier(
    n_estimators=250,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale,
    eval_metric='logloss'
)

model.fit(X_train, y_train)

# ---------------------------------------
# PREDICT
# ---------------------------------------

y_pred = model.predict(X_test)

# ---------------------------------------
# RESULTS
# ---------------------------------------

from sklearn.metrics import classification_report

print("\nTop-3 Prediction Report:\n")
print(classification_report(y_test, y_pred))


Top-3 Prediction Report:

              precision    recall  f1-score   support

           0       0.98      0.77      0.86      2259
           1       0.41      0.91      0.57       402

    accuracy                           0.79      2661
   macro avg       0.70      0.84      0.71      2661
weighted avg       0.89      0.79      0.82      2661



In [14]:
# ---------------------------------------
# 1. MERGE MAIN TABLES
# ---------------------------------------

df = results.merge(races, on="raceId")
df = df.merge(drivers, on="driverId")
df = df.merge(constructors, on="constructorId")

print("After main merge:", df.columns)

# ---------------------------------------
# 2. LOAD QUALIFYING
# ---------------------------------------

qualifying = pd.read_csv(os.path.join(DATA_PATH, "qualifying.csv"))
print("Qualifying columns:", qualifying.columns)

# Ensure numeric
qualifying['position'] = pd.to_numeric(qualifying['position'], errors='coerce')

# ---------------------------------------
# 3. MERGE QUALIFYING (SAFE)
# ---------------------------------------

df = df.merge(
    qualifying[['raceId', 'driverId', 'position']],
    on=['raceId', 'driverId'],
    how='left'
)

print("After qualifying merge:", df.columns)

# ---------------------------------------
# 4. RENAME + FIX
# ---------------------------------------

if 'position' in df.columns:
    df.rename(columns={'position': 'quali_pos'}, inplace=True)

# If still missing (fallback safety)
if 'quali_pos' not in df.columns:
    print("⚠️ quali_pos not found — creating fallback")
    df['quali_pos'] = 20

# Fill missing values
df['quali_pos'] = pd.to_numeric(df['quali_pos'], errors='coerce').fillna(20)

print("Final columns:", df.columns)

# ---------------------------------------
# 5. NOW SELECT COLUMNS (SAFE)
# ---------------------------------------

df = df[['raceId', 'year', 'round', 'driverId', 'constructorId',
         'grid', 'positionOrder', 'points', 'quali_pos']]

After main merge: Index(['resultId', 'raceId', 'driverId', 'constructorId', 'number_x', 'grid',
       'position', 'positionText', 'positionOrder', 'points', 'laps', 'time_x',
       'milliseconds', 'fastestLap', 'rank', 'fastestLapTime',
       'fastestLapSpeed', 'statusId', 'year', 'round', 'circuitId', 'name_x',
       'date', 'time_y', 'url_x', 'fp1_date', 'fp1_time', 'fp2_date',
       'fp2_time', 'fp3_date', 'fp3_time', 'quali_date', 'quali_time',
       'sprint_date', 'sprint_time', 'driverRef', 'number_y', 'code',
       'forename', 'surname', 'dob', 'nationality_x', 'url_y',
       'constructorRef', 'name_y', 'nationality_y', 'url'],
      dtype='object')
Qualifying columns: Index(['qualifyId', 'raceId', 'driverId', 'constructorId', 'number',
       'position', 'q1', 'q2', 'q3'],
      dtype='object')
After qualifying merge: Index(['resultId', 'raceId', 'driverId', 'constructorId', 'number_x', 'grid',
       'position_x', 'positionText', 'positionOrder', 'points', 'laps',
    

In [16]:
# Rename safely
if 'position_y' in df.columns:
    df.rename(columns={'position_y': 'quali_pos'}, inplace=True)
elif 'position' in df.columns:
    df.rename(columns={'position': 'quali_pos'}, inplace=True)

# Drop safely
df.drop(columns=['position_x'], errors='ignore', inplace=True)

# Ensure column exists
if 'quali_pos' not in df.columns:
    df['quali_pos'] = 20

# Convert to numeric
df['quali_pos'] = pd.to_numeric(df['quali_pos'], errors='coerce').fillna(20)

In [20]:
features = ['grid', 'quali_pos', 'driver_avg_pos', 'constructor_avg_pos']

In [18]:
from sklearn.metrics import classification_report, accuracy_score

y_pred = model.predict(X_test)

print("\n===== RESULTS =====")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


===== RESULTS =====
Accuracy: 0.7899285982713266

Classification Report:

              precision    recall  f1-score   support

           0       0.98      0.77      0.86      2259
           1       0.41      0.91      0.57       402

    accuracy                           0.79      2661
   macro avg       0.70      0.84      0.71      2661
weighted avg       0.89      0.79      0.82      2661



In [19]:
print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)

Training size: (23644, 3)
Testing size: (2661, 3)


In [21]:
import joblib

joblib.dump(model, "f1_model.pkl")
print("Model saved 😏")

Model saved 😏
